In [ ]:
#%%writefile C:\Users\neele\Music\Travscape\agents\planner.py
from agents.utils import get_today_str
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from agents.GlobalState import AgentState
from agents.planner_output import PlannerOutput
from agents.prompts import planner_message
from agents.utils import get_today_str

load_dotenv(override=True)

################Example:
trip_details = {
      "destination": "Tokyo",
      "dates": "October 10, 2025",
      "no._of_days": 5,
      "travelers": 3,
      "budget": "Budget trip",
      "purpose": "sightseeing",
      "preferences": None
    }
################

planner = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

planner_with_output = planner.with_structured_output(PlannerOutput)

#System message initialization with today's date
system_message = planner_message.format(
  date=get_today_str(),
  trip_details=trip_details
)

def Planner_node(state:AgentState) -> AgentState:
  
    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True

    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages

    try:
        response = planner_with_output.invoke(messages)

        if response.need_clarification and response.clarification_question is not None:
          ai_response = AIMessage(content=response.clarification_question)
          updated_messages = messages + [ai_response]

          return Command(
              update={
                  "messages": updated_messages
              }
          )

        else:
            plan_dict = response.plan.model_dump() if response.plan else {}
            ai_response = AIMessage(content=f"Plan generated: {plan_dict}")
            updated_messages = messages + [ai_response]

            return Command(
                update={
                    "messages": updated_messages,
                    "trip_plan": [plan_dict]
                }
            )
        
    except Exception as e:
        print(f"Error: {e}")
        error_message = AIMessage(content="Sorry, I encountered an error. Please try again.")
        return Command(
            update={"messages": messages + [error_message]}
        )
    
checkpointer = InMemorySaver()

planner_builder = StateGraph(AgentState)

planner_builder.add_node("Planner", Planner_node)

planner_builder.add_edge(START, "Planner")
planner_builder.add_edge("Planner", END)

graph = planner_builder.compile(checkpointer=checkpointer)



In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
import gradio as gr
config = {"configurable": {"thread_id": "1"}}

def chat(message, history):
    try:
        # Extract user message content
        user_message = message if isinstance(message, str) else message.get("text", "")
        
        # Invoke the graph
        result = graph.invoke(
            {"messages": [HumanMessage(content=user_message)]}, 
            config=config
        )
        
        # Return the assistant's response in proper format
        assistant_response = result["messages"][-1].content
        return assistant_response
        
    except Exception as e:
        print(f"Chat error: {e}")
        return (f"Sorry, I encountered an error. -> {e}")

gr.ChatInterface(
    chat, 
    type="messages",
    title="Trav - Your Travel Planning Assistant",
    description="Hi! I'm Trav, ready to help you plan your next adventure!"
).launch()

In [ ]:
#%%writefile C:\Users\neele\Music\Travscape\agents\orchestrator.py

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.types import Command
from agents.GlobalState import AgentState
from agents.orchestrator_output import OrchestratorDecision
from agents.prompts import orchestrator_message
from agents.utils import get_today_str
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv(override=True)

In [ ]:
import httpx
import os
from langchain.tools import tool
from typing import Optional, Dict, Any
from dotenv import load_dotenv

load_dotenv(override=True)

#Get City Code
def get_airport_code(location):
  """This function searches the source's and destination's airport ID for a given location using the Booking.com API. The first step when searching for flights."""

  url = "https://google-flights2.p.rapidapi.com/api/v1/searchAirport"

  querystring = {"query":location,"language_code":"en-US","country_code":"US"}

  headers = {
    "x-rapidapi-key": os.getenv('x-rapidapi-key'),
    "x-rapidapi-host": "google-flights2.p.rapidapi.com"
  }
  try:
    response = httpx.get(url, headers=headers, params=querystring)
    response.raise_for_status()  # Raise an error for bad responses
    airport_data = response.json().get('data', [])
    
    if not airport_data:
        print("No destinations found for this query.")
        airport_details = []

    airport_list = airport_data[0]["list"]
    airport_code = airport_list[0]["id"]
    #all_airports = [dest["list"] for dest in airport_code] #Saves all airport IDs for the given location
    #airport_code = [aid["id"] for aid in all_airports[0]] #Takes the first airport ID from the list
  
  except httpx.HTTPStatusError as e:
    print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
  except httpx.RequestError as e:
    print(f"Request error occurred: {e}") 
  
  return airport_code

def search_flights(
    departure_id: str,
    arrival_id: str,
    outbound_date: str,
    return_date: str,
    travel_class: str,
    adults: str,
    children: str,
    infants: str,
    show_hidden: str,
    currency: str,
    language_code: str,
    country_code: str,
    search_type: str,
) -> Dict[str, Any]:

    url = "https://google-flights2.p.rapidapi.com/api/v1/searchFlights"

    query = {
        "departure_id": departure_id,
        "arrival_id": arrival_id,
        "outbound_date": outbound_date,
        "travel_class": travel_class,
        "adults": adults,
        "children": children,
        "infant_on_lap": infants,
        "show_hidden": show_hidden,
        "currency": currency,
        "language_code": language_code,
        "country_code": country_code,
        "search_type": search_type,
    }

    # Only include return_date if provided
    if return_date and return_date.strip():
        query["return_date"] = return_date

    headers = {
        "x-rapidapi-key": os.getenv('x-rapidapi-key'),
        "x-rapidapi-host": "google-flights2.p.rapidapi.com",
    }

    try:
        resp = httpx.get(url, headers=headers, params=query, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        return {"error": str(e)}

    payload = resp.json()

    itineraries = payload.get("data", {}).get("itineraries", {}) or {}
    top_flights = itineraries.get("topFlights", []) or []
    other_flights = itineraries.get("otherFlights", []) or []

    # ---- Minimal LEG extractor ----
    def build_leg(leg: Dict[str, Any]) -> Dict[str, Any]:
        dep = leg.get("departure_airport") or {}
        arr = leg.get("arrival_airport") or {}
        dur = leg.get("duration") or {}

        return {
            "departure_airport_code": dep.get("airport_code"),
            "departure_airport_name": dep.get("airport_name"),
            "departure_time": dep.get("time"),
            "arrival_airport_code": arr.get("airport_code"),
            "arrival_airport_name": arr.get("airport_name"),
            "arrival_time": arr.get("time"),
            "leg_duration_text": dur.get("text"),
            "airline": leg.get("airline"),
            "airline_logo": leg.get("airline_logo"),
            "flight_number": leg.get("flight_number"),
        }

    # ---- Minimal Itinerary extractor ----
    def normalize_itinerary(itin: Dict[str, Any]) -> Dict[str, Any]:
        raw_flights = itin.get("flights")
        if raw_flights is None:
            flights_list = []
        elif isinstance(raw_flights, dict):
            flights_list = [raw_flights]
        elif isinstance(raw_flights, list):
            flights_list = raw_flights
        else:
            flights_list = []

        flights_min = [build_leg(f) for f in flights_list]

        return {
            "departure_time": itin.get("departure_time"),
            "arrival_time": itin.get("arrival_time"),
            "duration_text": (itin.get("duration") or {}).get("text"),
            "price": itin.get("price"),
            "stops": itin.get("stops"),
            "booking_token": itin.get("booking_token"),
            "flights": flights_min,
        }

    normalized_top = [normalize_itinerary(it) for it in top_flights]
    #normalized_other = [normalize_itinerary(it) for it in other_flights]

    return {
        "top_itineraries": normalized_top,
        #"other_itineraries": normalized_other,
    }

#@tool
def flight_search_tool(
    departure: str,
    arrival: str,
    outbound_date: str,
    return_date: str = "",
    travel_class: str = "ECONOMY",
    adults: str = "1",
    children: str = "0",
    infants: str = "0",
    show_hidden: str = "1",
    currency: str = "INR",
    language_code: str = "en-US",
    country_code: str = "IN",
    search_type: str = "best",
) -> Dict[str, Any]:
    
    """Combined tool to search for flights using the provided parameters."""
    departure_code = get_airport_code(departure)
    arrival_code = get_airport_code(arrival)
    print(departure_code, arrival_code)

    # if not departure_code or arrival_code:
    #     return {"error": "Could not find airport codes for the provided locations."}
    
    result= search_flights(
        departure_id=departure_code,
        arrival_id=arrival_code,
        outbound_date=outbound_date,
        return_date=return_date,
        travel_class=travel_class,
        adults=adults,
        children=children,
        infants=infants,
        show_hidden=show_hidden,
        currency=currency,
        language_code=language_code,
        country_code=country_code,
        search_type=search_type,
    )

    return result

In [ ]:
result = flight_search_tool(departure="Los Angeles",arrival="New York",outbound_date="2025-11-25",travel_class="ECONOMY",adults="1",currency="INR")
print(result)

In [1]:
from agents.utils import flight_search_tool
from dotenv import load_dotenv
load_dotenv(override=True)

result = flight_search_tool(departure="Los Angeles",arrival="New York",outbound_date="2025-11-25",travel_class="ECONOMY",adults="1",currency="INR")
print(result)


LAX JFK
{'top_itineraries': [{'departure_time': '25-11-2025 08:53 PM', 'arrival_time': '26-11-2025 05:13 AM', 'duration_text': '5 hr 20 min', 'price': 18975, 'stops': 0, 'booking_token': 'W1syLDEsWyIxIiwiMCIsIjAiLDBdLFsiQ2pSSVRHVlZRMFZIVmtkTWJHZEJSVTl1UjJkQ1J5MHRMUzB0TFMwdExYWjNZbWx5TjBGQlFVRkJSMnRtU1Y5QlJHMWxRVUZCRWdaR09USTFNRFFhQ3dpZmxBRVFBQm9EU1U1U09CMXdscWNCIl1dLFsiMjAyNS0xMS0yNSIsIkxBWCIsIkpGSyIsW1siTEFYIiwiMjAyNS0xMS0yNSIsIkpGSyIsIkY5IiwiMjUwNCJdXV1d', 'flights': [{'departure_airport_code': 'LAX', 'departure_airport_name': 'Los Angeles International Airport', 'departure_time': '2025-11-25 20:53', 'arrival_airport_code': 'JFK', 'arrival_airport_name': 'John F. Kennedy International Airport', 'arrival_time': '2025-11-26 05:13', 'leg_duration_text': '5 hr 20 min', 'airline': 'Frontier', 'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/F9.png', 'flight_number': 'F9 2504'}]}, {'departure_time': '25-11-2025 12:47 PM', 'arrival_time': '25-11-2025 09:05 PM', 'duration_

In [26]:
from dotenv import load_dotenv
import os
import httpx

load_dotenv(override=True)

def maps_text_search(query: str):
  """Tool for maps text search"""
  url = "https://places.googleapis.com/v1/places:searchText"
  
  params = {
    "textQuery": query
  }

  headers = {
    "X-Goog-Api-Key": os.getenv("GOOGLE_API_KEY"),
    "X-Goog-FieldMask": "places.name,places.displayName,places.formattedAddress,places.nationalPhoneNumber,places.internationalPhoneNumber,places.priceLevel,places.rating,places.googleMapsUri,places.websiteUri,places.regularOpeningHours,places.googleMapsLinks"
  }

  response = httpx.post(url, json=params, headers=headers)

  if response.status_code == 200:
    return response.json()
  else:
    print(f"Error {response.status_code}: {response.text}")
    return None

In [27]:
place = maps_text_search("Fushimi Inari Shrine")
print(place)

{'places': [{'name': 'places/ChIJIW0uPRUPAWAR6eI6dRzKGns', 'nationalPhoneNumber': '075-641-7331', 'internationalPhoneNumber': '+81 75-641-7331', 'formattedAddress': '68 Fukakusa Yabunouchichō, Fushimi Ward, Kyoto, 612-0882, Japan', 'rating': 4.6, 'googleMapsUri': 'https://maps.google.com/?cid=8870624639634301673&g_mp=Cidnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLlNlYXJjaFRleHQQAhgEIAA', 'websiteUri': 'https://inari.jp/', 'regularOpeningHours': {'openNow': True, 'periods': [{'open': {'day': 0, 'hour': 0, 'minute': 0}}], 'weekdayDescriptions': ['Monday: Open 24 hours', 'Tuesday: Open 24 hours', 'Wednesday: Open 24 hours', 'Thursday: Open 24 hours', 'Friday: Open 24 hours', 'Saturday: Open 24 hours', 'Sunday: Open 24 hours']}, 'displayName': {'text': 'Fushimi Inari Taisha', 'languageCode': 'en'}, 'googleMapsLinks': {'directionsUri': "https://www.google.com/maps/dir//''/data=!4m7!4m6!1m1!4e2!1m2!1m1!1s0x60010f153d2e6d21:0x7b1aca1c753ae2e9!3e0?g_mp=Cidnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLlNlYXJ

In [1]:
from datetime import datetime
import httpx
import os
import asyncio, aiohttp

async def general_search(queries: list[str] | str):
    """Tool for general web search. Accepts a single query string or a list of queries."""
    
    # Handle both single query and list of queries
    if isinstance(queries, str):
        queries = [queries]
    
    async def _search_single_query(session: aiohttp.ClientSession, query: str):
        """Helper function to search a single query using Tavily API"""
        url = "https://api.tavily.com/search"
        
        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 10,
            "topic": "general",
            "include_answer": True,
            "include_raw_content": False,
            "include_images": True
        }
        
        try:
            async with session.post(url, json=payload) as response:
                if response.status == 200:
                    return {
                        "query": query,
                        "success": True,
                        "data": await response.json()
                    }
                else:
                    text = await response.text()
                    print(f"Error {response.status} for query '{query}': {text}")
                    return {
                        "query": query,
                        "success": False,
                        "error": f"Status {response.status}: {text}"
                    }
        except Exception as e:
            print(f"Exception for query '{query}': {str(e)}")
            return {
                "query": query,
                "success": False,
                "error": str(e)
            }
    
    # Use a single session for all requests (more efficient)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[_search_single_query(session, query) for query in queries], return_exceptions=True)
    
    return results[0] if len(results) == 1 else results

In [2]:
result = await general_search(["What is the weather in New York?", "Cristiano ronaldo", "Did India qualify in FIFA 2026?", "Fushimi Inari Shrine"])
print(result)

[{'query': 'What is the weather in New York?', 'success': True, 'data': {'query': 'What is the weather in New York?', 'follow_up_questions': None, 'answer': 'The weather in New York in December 2025 is expected to be cold, with temperatures around 41°F to 33°F and occasional rain. The month typically has a high risk of precipitation. December is generally cold in New York.', 'images': ['https://cdn.apwx.net/img/maps/us/weather/states/ny/us-new-york-temps-c.jpg', 'https://media.foxweather.com/weather/NYC_Forecast.png', 'https://images.foxtv.com/static.fox5ny.com/www.fox5ny.com/content/uploads/2024/10/932/524/prec.jpg?ve=1&tl=1', 'https://www.anytraveltips.com/wp-content/uploads/new-york-december_weather1_x.jpg', 'https://cdn.abcotvs.com/dip/images/15438384_Winter-weather-2025-outlook.jpg'], 'results': [{'title': 'Weather in New York', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'New York', 'region': 'New York', 'country': 'United States of America', 'lat': 40

In [3]:
import httpx
import os
import asyncio
from typing import Dict, Any
from dotenv import load_dotenv

load_dotenv(override=True)

#Get City Code (Async Version)
async def get_airport_code(location):
    """This function searches the source's and destination's airport ID for a given location using the Booking.com API. The first step when searching for flights."""
    
    url = "https://google-flights2.p.rapidapi.com/api/v1/searchAirport"
    
    querystring = {"query": location, "language_code": "en-US", "country_code": "US"}
    
    headers = {
        "x-rapidapi-key": os.getenv('x-rapidapi-key'),
        "x-rapidapi-host": "google-flights2.p.rapidapi.com"
    }
    
    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(url, headers=headers, params=querystring)
            response.raise_for_status()  # Raise an error for bad responses
            airport_data = response.json().get('data', [])
            
            if not airport_data:
                print("No destinations found for this query.")
                return None
            
            airport_list = airport_data[0]["list"]
            airport_code = airport_list[0]["id"]
            return airport_code
    
    except httpx.HTTPStatusError as e:
        print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
        return None
    except httpx.RequestError as e:
        print(f"Request error occurred: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None


async def search_flights(
    departure_id: str,
    arrival_id: str,
    outbound_date: str,
    return_date: str,
    travel_class: str,
    adults: str,
    children: str,
    infants: str,
    show_hidden: str,
    currency: str,
    language_code: str,
    country_code: str,
    search_type: str,
) -> Dict[str, Any]:

    url = "https://google-flights2.p.rapidapi.com/api/v1/searchFlights"

    query = {
        "departure_id": departure_id,
        "arrival_id": arrival_id,
        "outbound_date": outbound_date,
        "travel_class": travel_class,
        "adults": adults,
        "children": children,
        "infant_on_lap": infants,
        "show_hidden": show_hidden,
        "currency": currency,
        "language_code": language_code,
        "country_code": country_code,
        "search_type": search_type,
    }

    # Only include return_date if provided
    if return_date and return_date.strip():
        query["return_date"] = return_date

    headers = {
        "x-rapidapi-key": os.getenv('x-rapidapi-key'),
        "x-rapidapi-host": "google-flights2.p.rapidapi.com",
    }

    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(url, headers=headers, params=query, timeout=15)
            resp.raise_for_status()
    except Exception as e:
        return {"error": str(e)}

    payload = resp.json()

    itineraries = payload.get("data", {}).get("itineraries", {}) or {}
    top_flights = itineraries.get("topFlights", []) or []
    other_flights = itineraries.get("otherFlights", []) or []

    # ---- Minimal LEG extractor ----
    def build_leg(leg: Dict[str, Any]) -> Dict[str, Any]:
        dep = leg.get("departure_airport") or {}
        arr = leg.get("arrival_airport") or {}
        dur = leg.get("duration") or {}

        return {
            "departure_airport_code": dep.get("airport_code"),
            "departure_airport_name": dep.get("airport_name"),
            "departure_time": dep.get("time"),
            "arrival_airport_code": arr.get("airport_code"),
            "arrival_airport_name": arr.get("airport_name"),
            "arrival_time": arr.get("time"),
            "leg_duration_text": dur.get("text"),
            "airline": leg.get("airline"),
            "airline_logo": leg.get("airline_logo"),
            "flight_number": leg.get("flight_number"),
        }

    # ---- Minimal Itinerary extractor ----
    def normalize_itinerary(itin: Dict[str, Any]) -> Dict[str, Any]:
        raw_flights = itin.get("flights")
        if raw_flights is None:
            flights_list = []
        elif isinstance(raw_flights, dict):
            flights_list = [raw_flights]
        elif isinstance(raw_flights, list):
            flights_list = raw_flights
        else:
            flights_list = []

        flights_min = [build_leg(f) for f in flights_list]

        return {
            "departure_time": itin.get("departure_time"),
            "arrival_time": itin.get("arrival_time"),
            "duration_text": (itin.get("duration") or {}).get("text"),
            "price": itin.get("price"),
            "stops": itin.get("stops"),
            "booking_token": itin.get("booking_token"),
            "flights": flights_min,
        }

    normalized_top = [normalize_itinerary(it) for it in top_flights]
    #normalized_other = [normalize_itinerary(it) for it in other_flights]

    return {
        "top_itineraries": normalized_top,
        #"other_itineraries": normalized_other,
    }


# @tool(args_schema=FlightSearchInput)
async def flight_search_tool(
    departure: str,
    arrival: str,
    outbound_date: str,
    return_date: str,
    travel_class: str,
    adults: str,
    children: str,
    infants: str,
    currency: str,
    search_type: str,
    show_hidden: str = "1",
    language_code: str = "en-US",
    country_code: str = "IN",
) -> Dict[str, Any]:
    
    """Combined tool to search for flights using the provided parameters."""
    
    # Concurrent airport code lookup - 2x faster!
    departure_code, arrival_code = await asyncio.gather(
        get_airport_code(departure),
        get_airport_code(arrival)
    )
    
    print(departure_code, arrival_code)

    # Validate we got both codes
    if not departure_code or not arrival_code:
        return {"error": "Could not find airport codes for the provided locations."}
    
    # Search flights with the obtained codes
    result = await search_flights(
        departure_id=departure_code,
        arrival_id=arrival_code,
        outbound_date=outbound_date,
        return_date=return_date,
        travel_class=travel_class,
        adults=adults,
        children=children,
        infants=infants,
        show_hidden=show_hidden,
        currency=currency,
        language_code=language_code,
        country_code=country_code,
        search_type=search_type,
    )

    return result

In [5]:
flights = await flight_search_tool( 
        departure="New Delhi",
        arrival="San Francisco",
        outbound_date="2025-12-15",
        return_date="",
        travel_class="ECONOMY",
        adults=1,
        children=0,
        infants=0,
        show_hidden="1",
        currency="INR",
        language_code="en-US",
        country_code="IN",
        search_type="best",)

print(flights)


DEL SFO
{'top_itineraries': [{'departure_time': '15-12-2025 05:00 AM', 'arrival_time': '15-12-2025 02:40 PM', 'duration_text': '23 hr 10 min', 'price': 46246, 'stops': 0, 'booking_token': 'W1syLDEsWyIxIiwiMCIsIjAiLDBdLFsiQ2pSSVVFeG5NMlpIUmtZMkxXTkJRVTVPT1dkQ1J5MHRMUzB0TFMwdExYQm1kbUV4TVVGQlFVRkJSMnMxYzJWRlRuZGlVMnRCRWdwV1V6TXdNM3hXVXpFNUdnc0lwdWtDRUFBYUEwbE9VamdkY1BtUkF3PT0iXV0sWyIyMDI1LTEyLTE1IiwiREVMIiwiU0ZPIixbWyJERUwiLCIyMDI1LTEyLTE1IiwiTEhSIiwiVlMiLCIzMDMiXSxbIkxIUiIsIjIwMjUtMTItMTUiLCJTRk8iLCJWUyIsIjE5Il1dXV0=', 'flights': [{'departure_airport_code': 'DEL', 'departure_airport_name': 'Indira Gandhi International Airport', 'departure_time': '2025-12-15 05:00', 'arrival_airport_code': 'LHR', 'arrival_airport_name': 'Heathrow Airport', 'arrival_time': '2025-12-15 09:30', 'leg_duration_text': '10 hr 0 min', 'airline': 'Virgin Atlantic', 'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/VS.png', 'flight_number': 'VS 303'}, {'departure_airport_code': 'LHR', 'departure_